# OLAP Data Analysis

This notebook applies classic **OLAP (Online Analytical Processing)** operations to the shipment dataset.

| Operation | Description |
|---|---|
| **Slice** | Filter rows on a single dimension (e.g., Warehouse A only) |
| **Dice** | Filter rows on multiple dimensions simultaneously |
| **Roll-Up** | Aggregate (group-by + mean/sum) across a dimension |
| **Drill-Down** | Narrow from a high-level group into a specific subset |
| **Pivot** | Cross-tabulate two dimensions with an aggregated metric |

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Dataset

In [ ]:
df = pd.read_csv("Train.csv")

print("Dataset Shape:", df.shape)
df.head()

## 3. Data Overview

In [ ]:
df.info()
df.describe()

## 4. Slice

Filter along a **single dimension** — equivalent to fixing one face of the OLAP cube.

### 4.1 Slice — Warehouse Block A

In [ ]:
slice_warehouse_A = df[df['Warehouse_block'] == 'A']
slice_warehouse_A.head()

### 4.2 Slice — On-Time Deliveries Only

In [ ]:
slice_delivery = df[df['Reached.on.Time_Y.N'] == 1]
slice_delivery.head()

## 5. Dice

Filter along **multiple dimensions simultaneously** — a sub-cube of the data.

In [ ]:
dice_data = df[
    (df['Warehouse_block'] == 'A') &
    (df['Mode_of_Shipment'] == 'Flight') &
    (df['Product_importance'] == 'high')
]

dice_data.head()

## 6. Roll-Up

Aggregate metrics by summarising over a dimension — moving **up** the hierarchy.

### 6.1 Roll-Up by Warehouse Block

In [ ]:
rollup_warehouse = df.groupby('Warehouse_block').agg({
    'Cost_of_the_Product': 'mean',
    'Discount_offered':    'mean',
    'Weight_in_gms':       'mean',
    'Reached.on.Time_Y.N': 'mean'
})

rollup_warehouse

### 6.2 Roll-Up by Shipment Mode — Delivery Rate

In [ ]:
rollup_shipment = df.groupby('Mode_of_Shipment')['Reached.on.Time_Y.N'].mean()
rollup_shipment

### 6.3 Roll-Up — Delivery Rate by Warehouse

In [ ]:
df.groupby('Warehouse_block')['Reached.on.Time_Y.N'].mean()

## 7. Roll-Up Visualisations

### 7.1 Warehouse Distribution — Pie Chart

In [ ]:
plt.figure(figsize=(6, 6))
plt.pie(
    df['Warehouse_block'].value_counts(),
    labels=df['Warehouse_block'].value_counts().index,
    autopct='%1.1f%%'
)
plt.title("Warehouse Distribution")
plt.tight_layout()
plt.show()

### 7.2 Average Product Cost by Warehouse — Line Plot

In [ ]:
warehouse_cost = df.groupby('Warehouse_block')['Cost_of_the_Product'].mean()

plt.figure(figsize=(7, 4))
plt.plot(warehouse_cost.index, warehouse_cost.values, marker='o')
plt.title("Average Product Cost by Warehouse (Roll-Up)")
plt.ylabel("Avg Cost")
plt.xlabel("Warehouse Block")
plt.tight_layout()
plt.show()

## 8. Drill-Down

Start from the rolled-up view and **descend** into a specific segment for deeper inspection.

In [ ]:
drilldown_A = df[df['Warehouse_block'] == 'A']
drilldown_A.head()

In [ ]:
# Drill further: delivery rate per shipment mode within Warehouse A
delivery_summary = df.groupby(['Warehouse_block', 'Mode_of_Shipment'])['Reached.on.Time_Y.N'].mean()
delivery_summary

## 9. Pivot — Delivery Performance Heatmap

Cross-tabulate `Warehouse_block` × `Mode_of_Shipment` with average on-time delivery rate.

In [ ]:
pivot_heatmap = df.pivot_table(
    values='Reached.on.Time_Y.N',
    index='Warehouse_block',
    columns='Mode_of_Shipment',
    aggfunc='mean'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot_heatmap, annot=True, cmap='Blues', fmt='.3f')
plt.title("Delivery Performance Heatmap (Warehouse × Shipment Mode)")
plt.tight_layout()
plt.show()